In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt
from scipy.stats import skew, kurtosis

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Setup DEAP

### Identify the correct path

The previous commands failed because the directory `/content/gdrive/Shareddrives/GCPDS/databases/DEAP/` could not be found.

To help you identify the correct path, I will list the contents of `/content/gdrive/Shareddrives/`. Please examine the output and replace the incorrect path with the correct one in the subsequent cells.

In [ ]:
from google.colab import drive
import requests
import os

drive.mount('/content/gdrive/')

In [ ]:
target_dir = '/content/gdrive/MyDrive/'

## Unzip

In [ ]:
if os.path.exists(target_dir):
    os.chdir(target_dir)
    print(f"Changed directory to: {os.getcwd()}")

else:
    print(f"Error: Directory not found: {target_dir}. Please ensure Google Drive is mounted and the path is correct.")

In [ ]:
if not os.path.exists('deap-dataset'):
  !unzip deap-dataset.zip
  print("deap-dataset.zip extracted.")

else:
  print("deap-dataset directory already exists. Skipping extraction.")

The `deap-dataset.zip` archive has been extracted, creating a directory named `deap-dataset/` which contains all the dataset files and subdirectories, including `data_preprocessed_python/`.

## load sub

The videos are in the order of Experiment_id, so not in the order of presentation. This means the first video is the same for each participant.

In [ ]:
target_dir = 'deap-dataset/data_preprocessed_python/'

In [ ]:
if os.path.exists(target_dir):
    os.chdir(target_dir)
    print(f"Changed directory to: {os.getcwd()}")

else:
    print(f"Error: Directory not found: {os.path.join(os.getcwd(), target_dir)}. Please ensure 'deap-dataset.zip' was extracted correctly and contains 'data_preprocessed_python/'.")

We've now navigated into the `deap-dataset/data_preprocessed_python/` directory, where the individual subject data files (`sXX.dat`) are located.

In [ ]:
i = 1
with open('s'+ '01' + '.dat', 'rb') as file:
  subject = pickle.load(file, encoding='latin1')

In [ ]:
def load_subject_data(subject_id_str):
  """Loads data for a specific subject from a .dat file."""
  file_name = f's{subject_id_str}.dat'
  with open(file_name, 'rb') as file:
    subject = pickle.load(file, encoding='latin1')
  return subject

# Example usage: Load data for subject '01'
subject_id = '01'
subject = load_subject_data(subject_id)
print(f"Successfully loaded data for subject {subject_id}.")

In [ ]:
subject.keys()

The `subject` dictionary contains two main keys: 'labels' and 'data'.

*   **'labels'**: This typically contains the self-assessment scores (valence, arousal, dominance, liking) for each video watched by the participant.
*   **'data'**: This holds the physiological signal data (e.g., EEG, EOG, EMG, GSR, respiration, temperature) recorded during the experiment.

In [ ]:
subject['data'].shape

The shape of `subject['data']` is `(40, 40, 8064)`:

*   `40`: Represents the 40 experimental trials (videos) for this participant.
*   `40`: Represents the 40 physiological channels (e.g., 32 EEG, 8 peripheral).
*   `8064`: Represents the number of data points (samples) per channel per trial. Given a sampling rate of 128 Hz and a trial duration of 63 seconds, this corresponds to `128 samples/second * 63 seconds = 8064 samples`.

In [ ]:
subject['labels'].shape

The shape of `subject['labels']` is `(40, 4)`:

*   `40`: Represents the 40 experimental trials (videos).
*   `4`: Represents the 4 self-assessment scores: Valence, Arousal, Dominance, and Liking, typically rated on a scale of 1-9.

## Plotting

In [ ]:
def plot_classifier_metrics(*rows, title="Classifier Comparison") -> None:
    def _resolve_name(r):
        idx = r[1]
        if isinstance(idx, (int, np.integer)):
            return classifier_labels[idx - 1]
        return idx

    names = [_resolve_name(r) for r in rows]

    metrics = {
        "Accuracy": [r[2] for r in rows],
        "Precision": [r[3] for r in rows],
        "Recall": [r[4] for r in rows],
        "F1": [r[5] for r in rows],
    }

    x = np.arange(len(rows))
    width = 0.2

    plt.figure(figsize=(10, 5))

    for i, (metric, values) in enumerate(metrics.items()):
        plt.bar(x + i * width, values, width=width, label=metric)

    plt.xticks(x + width * 1.5, names)
    plt.title(title)
    plt.xlabel("Classifier")
    plt.ylabel("Score")
    plt.legend()
    plt.grid(True)
    plt.show()

## Evaluation Metrics

Computes accuracy, precision, recall, and F1 from the confusion matrix. Every classifier below is scored with this same function, so results are directly comparable.

In [ ]:
def evaluate_metrics(y_true, y_pred) -> tuple:

    cm = confusion_matrix(y_true, y_pred)

    if cm.shape[0] < 2:
        return np.nan, np.nan, np.nan, np.nan

    TN = cm[0,0] 
    FP = cm[0,1]
    FN = cm[1,0]
    TP = cm[1,1]

    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP + np.finfo(float).eps)
    recall = TP / (TP + FN + np.finfo(float).eps)

    f1 = 2 * (precision * recall) / \
         (precision + recall + np.finfo(float).eps)

    return accuracy, precision, recall, f1

## Feature Extraction

For a single trial, `runTrial()`:

1. Band-pass filters all 32 EEG channels (using the filter defined in Settings).
2. Keeps only the current 15-second segment.
3. Computes 6 statistical features per channel — mean, std, variance, RMS, skewness, kurtosis.

It returns the resulting feature vector together with the trial's label, and relies on `data`, `trial`, `segment`, and `current_labels` being set by the loop that calls it.

In [ ]:
def extract(eeg_seg) -> []:
    feature_vector = []
    
    for ch in range(32):

        x = eeg_seg[ch,:]

        mean_val = np.mean(x)
        std_val = np.std(x)
        var_val = np.var(x)
        rms_val = np.sqrt(np.mean(x**2))

        skew_val = skew(x)
        kurt_val = kurtosis(x)

        feature_vector.extend([
            mean_val,
            std_val,
            var_val,
            rms_val,
            skew_val,
            kurt_val
        ])
    
    return feature_vector

In [ ]:
def runTrial() -> []:
    eeg = data[trial,0:32,:]

    # FILTERING

    eeg_filt = np.zeros_like(eeg)

    for ch in range(32):

        eeg_filt[ch,:] = filtfilt(
            b,
            a,
            eeg[ch,:]
        )

    
    # CURRENT SEGMENT

    start_idx = segment * segment_length
    end_idx = (segment + 1) * segment_length

    eeg_seg = eeg_filt[:,start_idx:end_idx]


    # FEATURE EXTRACTION
    feature_vector = extract(eeg_seg)
    
    return [feature_vector, current_labels[trial]]

# Settings

All global constants and variables used, main ones include: 
1. `Fs`: sampling rate/frequency (128 Hz).

2. `segment_length` and `segment_labels`: each 60-second trial is split into four 15-second windows.

3. `label_column`: zero-based index of self-assessment scores to classify (0 = Valence).

In [ ]:
# FILTER

Fs = 128 # Hz

b, a = butter(
    4, # 4th-order
    [0.5/(Fs/2), 45/(Fs/2)], # [0.5–45 Hz], Typical EEG range
    btype='bandpass'
)


# SEGMENTS

# 
segment_length = 15 * Fs
segment_splits = []

segment_labels = [
    "0-15 s",
    "15-30 s",
    "30-45 s",
    "45-60 s"
]


# Classifiers

classifier_labels = [
    "SVM",
    "KNN",
    "Logistic Regression",
    "Decision Tree",
    "LDA"
]

label_column = 0
results = []


# Dataset loading

For each of the 4 segments: loads every subject's `.dat` file, extracts features for all 40 trials via `runTrial()`, then standardizes the resulting feature matrix `X` with `StandardScaler`. After this cell runs, `X` and `Y` hold the data for the last segment processed.

In [ ]:
def pca(XTrain, XTest, n_comp=20) -> tuple:
    Xtrain, pca_model = PCA(XTrain,n_components=n_comp)
    Xtest = pca_model.transform(Xtest)
    
    return XTrain, XTest

def split_traintest(X_red, Y) -> tuple:
    Xtrain, Xtest, Ytrain, Ytest = train_test_split(
        X_red, Y, test_size=0.30, random_state=42, stratify=Y
    )
    
    return Xtrain, Xtest, Ytrain, Ytest

In [ ]:
def cache_data() -> None:
    # CACHE ALL SUBJECT DATA (load .dat files ONCE)

    all_subject_data = []
    all_subject_labels = []

    for subject in range(1, 33):
        filename = f"s{subject:02d}.dat"
        try:
            with open(filename, "rb") as f:
                dataset = pickle.load(f, encoding="latin1")
            all_subject_data.append(dataset["data"])
            all_subject_labels.append(dataset["labels"])
        except (EOFError, pickle.UnpicklingError, OSError) as e:
            print(f"WARNING: skipping s{subject:02d}.dat (corrupted: {e})")
            continue

    print(f"Cached {len(all_subject_data)} subjects successfully.")

In [ ]:
def preprocess() -> None:
    # PREPROCESS ALL SEGMENTS — Feature Extraction + PCA + Split
    for segment in range(4):

        print("\n")
        print("="*40)
        print(f"SEGMENT {segment+1}")
        print("="*40)

        X = []
        Y = []

        for subj_idx in range(len(all_subject_data)):

            data = all_subject_data[subj_idx]
            labels = all_subject_labels[subj_idx]

            current_labels = labels[:, label_column]
            current_labels = np.where(current_labels > 5, 1, 0)

            for trial in range(40):

                # Extract EEG channels (first 32 of 40)
                eeg = data[trial, 0:32, :]

                # Bandpass filter: 0.5-45 Hz, 4th-order Butterworth
                eeg_filt = np.zeros_like(eeg)
                for ch in range(32):
                    eeg_filt[ch, :] = filtfilt(b, a, eeg[ch, :])

                # Slice current segment
                start_idx = segment * segment_length
                end_idx = (segment + 1) * segment_length
                eeg_seg = eeg_filt[:, start_idx:end_idx]

                # Extract 6 statistical features per channel
                feature_vector = []
                for ch in range(32):
                    x = eeg_seg[ch, :]
                    feature_vector.extend([
                        np.mean(x),
                        np.std(x),
                        np.var(x),
                        np.sqrt(np.mean(x**2)),
                        skew(x),
                        kurtosis(x)
                    ])

                X.append(feature_vector)
                Y.append(current_labels[trial])
            
        # NORMALISATION
        X = np.array(X)
        Y = np.array(Y)
        
        # TRAIN / TEST SPLIT
        Xtrain, Xtest, Ytrain, Ytest = split_traintest(X, Y)

        scaler = StandardScaler()
        XTrain = scaler.fit_transform(XTrain)
        Xtest = scaler.transform(XTest)
            
        # PCA
        XTrain, Xtest = pca(XTrain, XTest, 20)
        

        segment_splits.append((Xtrain, Xtest, Ytrain, Ytest))

    print("Preprocessing complete. segment_splits has 4 entries.")

# Classifiers
---

Five classifiers are trained on the same `Xtrain`/`Ytrain` split and compared on the same `Xtest`/`Ytest`. Each one lives in its own cell below: it trains, predicts, scores itself with `evaluate_metrics`, and prints its own result — so you can run just one cell to see just that classifier's result, or run the whole notebook to get all of them.

## SVM (Support Vector Model)

Finds the hyperplane (here, via an RBF kernel) that best separates the two valence classes, maximizing the margin between them.

In [ ]:
def run_svm(Xtrain, Xtest, Ytrain, Ytest, segment):
    model = SVC(kernel='rbf')
    model.fit(Xtrain, Ytrain)

    Ypred = model.predict(Xtest)
    acc, prec, rec, f1 = evaluate_metrics(Ytest, Ypred)

    return [segment+1, 1, acc, prec, rec, f1]


In [ ]:
for segment in range(4):
    Xtrain, Xtest, Ytrain, Ytest = segment_splits[segment]
    result = run_svm(Xtrain, Xtest, Ytrain, Ytest, segment)
    results.append(result)
    
    print(f"\n--- SVM (Segment {segment+1}) ---")
    print(f"Accuracy:  {result[2]:.4f}")
    print(f"Precision: {result[3]:.4f}")
    print(f"Recall:    {result[4]:.4f}")
    print(f"F1:        {result[5]:.4f}")
    plot_classifier_metrics(result, title=f"SVM (Segment {segment+1})")

## KNN (k-th Nearest Neighbor)

Classifies each test sample by majority vote among its 5 nearest neighbors in the training set (Euclidean distance in the 20-component PCA space).

In [ ]:
def run_knn(Xtrain, Xtest, Ytrain, Ytest, segment):
    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(Xtrain, Ytrain)

    Ypred = model.predict(Xtest)
    acc, prec, rec, f1 = evaluate_metrics(Ytest, Ypred)

    return [segment+1, 2, acc, prec, rec, f1]


In [ ]:
for segment in range(4):
    Xtrain, Xtest, Ytrain, Ytest = segment_splits[segment]
    result = run_knn(Xtrain, Xtest, Ytrain, Ytest, segment)
    results.append(result)
    print(f"\n--- KNN (Segment {segment+1}) ---")
    print(f"Accuracy:  {result[2]:.4f}")
    print(f"Precision: {result[3]:.4f}")
    print(f"Recall:    {result[4]:.4f}")
    print(f"F1:        {result[5]:.4f}")
    plot_classifier_metrics(result, title=f"KNN (Segment {segment+1})")

## Logistic Regression

Fits a linear decision boundary by modeling the log-odds of the positive class as a linear combination of the input features.

In [ ]:
def run_logistic_regression(Xtrain, Xtest, Ytrain, Ytest, segment):
    model = LogisticRegression(max_iter=5000)
    model.fit(Xtrain, Ytrain)

    Ypred = model.predict(Xtest)
    acc, prec, rec, f1 = evaluate_metrics(Ytest, Ypred)

    return [segment+1, 3, acc, prec, rec, f1]


In [ ]:
for segment in range(4):
    Xtrain, Xtest, Ytrain, Ytest = segment_splits[segment]
    result = run_logistic_regression(Xtrain, Xtest, Ytrain, Ytest, segment)
    results.append(result)
    print(f"\n--- Logistic Regression (Segment {segment+1}) ---")
    print(f"Accuracy:  {result[2]:.4f}")
    print(f"Precision: {result[3]:.4f}")
    print(f"Recall:    {result[4]:.4f}")
    print(f"F1:        {result[5]:.4f}")
    plot_classifier_metrics(result, title=f"Logistic Regression (Segment {segment+1})")

## Decision Tree (DecTree)

Learns a sequence of feature thresholds that split the training data into increasingly pure groups, forming a tree whose leaves predict a class.

In [ ]:
def run_decision_tree(Xtrain, Xtest, Ytrain, Ytest, segment):
    model = DecisionTreeClassifier()
    model.fit(Xtrain, Ytrain)

    Ypred = model.predict(Xtest)
    acc, prec, rec, f1 = evaluate_metrics(Ytest, Ypred)

    return [segment+1, 4, acc, prec, rec, f1]


In [ ]:
for segment in range(4):
    Xtrain, Xtest, Ytrain, Ytest = segment_splits[segment]
    result = run_decision_tree(Xtrain, Xtest, Ytrain, Ytest, segment)
    results.append(result)
    print(f"\n--- Decision Tree (Segment {segment+1}) ---")
    print(f"Accuracy:  {result[2]:.4f}")
    print(f"Precision: {result[3]:.4f}")
    print(f"Recall:    {result[4]:.4f}")
    print(f"F1:        {result[5]:.4f}")
    plot_classifier_metrics(result, title=f"Decision Tree (Segment {segment+1})")

## LDA (Linear Discriminant Analysis)

 Projects the data onto the direction that best separates the two classes by maximizing between-class variance relative to within-class variance.

In [ ]:
def run_lda(Xtrain, Xtest, Ytrain, Ytest, segment):
    model = LinearDiscriminantAnalysis()
    model.fit(Xtrain, Ytrain)

    Ypred = model.predict(Xtest)
    acc, prec, rec, f1 = evaluate_metrics(Ytest, Ypred)

    return [segment+1, 5, acc, prec, rec, f1]


In [ ]:
for segment in range(4):
    Xtrain, Xtest, Ytrain, Ytest = segment_splits[segment]
    result = run_lda(Xtrain, Xtest, Ytrain, Ytest, segment)
    results.append(result)
    print(f"\n--- LDA (Segment {segment+1}) ---")
    print(f"Accuracy:  {result[2]:.4f}")
    print(f"Precision: {result[3]:.4f}")
    print(f"Recall:    {result[4]:.4f}")
    print(f"F1:        {result[5]:.4f}")
    plot_classifier_metrics(result, title=f"LDA (Segment {segment+1})")

# Results
---

Collects every classifier's result from `results` into a single table, reports the best model by F1 score, and plots a bar chart comparing F1 across all of them.

In [ ]:
result_table = pd.DataFrame(
    results,
    columns=[
        "Segment",
        "Classifier",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]
)

display_table = result_table.copy()
display_table["Classifier"] = display_table["Classifier"].map(
    lambda classifier_id: classifier_labels[classifier_id - 1]
)
print(display_table)

plot_classifier_metrics(*result_table.values, title="All Classifiers — Summary")

In [ ]:
print(display_table)

# Best model
best_index = result_table["F1"].idxmax()

print("\nBEST MODEL\n")
print(display_table.loc[best_index])
